# Does structure exist in the EUR/USD exchange rate?

### A pre-registered search, and what a ten-parameter model found that a neural network did not

**Author:** Velizar Mitov  **Institution:** SoftUni — Deep Learning  **Date:** August 2026

---

## Abstract

This project asks whether next-day EUR/USD returns contain learnable structure, and answers
it with a pre-registered hypothesis registry rather than a single model. Fifty-six
hypotheses were registered across fifteen families; **thirty-seven are recorded as
rejected.** Directional prediction remains at ROC-AUC ≈ 0.50, consistent with Meese and
Rogoff (1983).

The substantive finding is not a better neural network. It is that a **ten-parameter
calendar model** — GARCH(1,1) multiplied by six weekday factors — outperforms a trained
five-seed multi-task LSTM ensemble on the volatility target, on both the validation and
test blocks of an identical row set. Three exotic architectures (a liquid time-constant
network, a spiking abstention readout, and a discrete-Hodge microstructure estimator) were
built, tested, and honestly rejected.

The methodological contribution is the apparatus: Bonferroni-corrected family bars,
permutation and initialisation nulls, block bootstraps, mandatory replication across
instruments, power analysis before spending significance, and an append-only registry in
which failures are recorded as prominently as successes.

---

## Reader's guide

| Section | Content |
|---|---|
| 1 | Introduction — the efficiency problem |
| 2 | Problem statement and economic framing |
| 3 | Previous research and how this work differs |
| 4 | Data acquisition, cleaning and validity |
| 5 | Methodology and architectures |
| 6 | Testing protocol |
| 7 | Results and visualisation |
| 8 | Conclusion |
| A | Rubric map, reproducibility, limitations |

Supporting documents: `ARCHITECTURE_DOCS.md` (technical reference), `IMPROVEMENT_LOG.md`
(dated research journal), `notebooks/01_data_preparation.ipynb` (full pipeline and Section 22
change record), and fifteen `results/*_hypothesis_log.csv` registry files.

---

# 1. Introduction

In 1983 Richard Meese and Kenneth Rogoff published a result that has shaped exchange-rate
economics ever since: no structural model of the exchange rate they tested could beat a
**random walk** out of sample. Four decades of subsequent literature has largely confirmed
it. Any project that proposes to forecast EUR/USD is therefore proposing to overturn one of
the most robust negative results in empirical finance.

This project began, as most do, by assuming that sufficiently sophisticated machinery would
find what simpler methods had missed. It ends somewhere more interesting.

Over fifty-six registered hypotheses, the pattern that emerged was consistent and initially
unwelcome. Each time a complex model appeared to find structure, decomposition revealed the
same thing underneath: **the calendar.** A multi-task LSTM ensemble that beat GARCH turned
out to be exploiting the Friday-to-Monday weekend gap. A spiking neural readout that
appeared to identify its own confident moments turned out to be firing during the Asian
session, when the market is quiet. In each case a lookup table encoded the same effect more
accurately than the network that discovered it.

This is not a story about failure. It is a story about **measurement discipline** — about
building an apparatus honest enough to tell you when your best idea is an artifact, and
then believing it.

---

# 2. Problem statement

## 2.1 The applied problem

A retail participant holding EUR/USD exposure must answer two operationally distinct
questions each day:

1. **Direction** — will the rate rise or fall? This determines position sign.
2. **Magnitude** — how large will the move be? This determines position *size*, stop
   placement and risk limits.

The second question is economically valuable even when the first is unanswerable. A trader
who cannot predict direction but can predict volatility can still size positions correctly,
avoid over-leverage before high-variance sessions, and set stops that are neither too tight
nor wastefully wide. **This project addresses both, and reports honestly that only the
second admits a useful answer.**

## 2.2 Why calendar time is the wrong clock

Financial data arrives on a calendar, but information does not. A Friday close and the
following Monday open are separated by roughly sixty-five hours of wall-clock time and
perhaps ten hours of market activity. A model trained on uniformly-spaced daily bars treats
those two intervals as equivalent.

Clark (1973) formalised this: prices follow a Brownian motion **subordinated** to a
stochastic clock driven by transaction arrivals. Returns are approximately Gaussian in
business time and heavy-tailed in calendar time. Every calendar-time model therefore carries
a specification error that a naive fit will silently absorb — and, as this project
discovered, will then mistake for skill.

## 2.3 The economic threshold that governs every claim

A statistical edge is not a trading edge. Measured on the broker feed used here, the median
EUR/USD spread is **0.46 basis points** one-way, or roughly **0.92 bp** per round trip. Any
predictive improvement smaller than that is economically void regardless of its p-value.

This constraint was applied as a hard filter and it terminated an entire research line: the
best selective-prediction gain achieved was **0.29 bp**, against a **0.92 bp** cost — a
ratio of 0.32×. The model would have lost money reliably even if every statistical claim
about it were true.

## 2.4 Formal statement

Let $P_t$ be the daily closing rate and $r_t = 100 \cdot \log(P_t / P_{t-1})$ the percentage
log return. We define two targets:

$$y^{\text{dir}}_{t} = \mathbb{1}[r_{t+1} > 0], \qquad y^{\text{vol}}_{t} = |r_{t+1}|$$

and seek $\hat{f}$ minimising out-of-sample loss against pre-registered benchmarks, subject
to the constraint that any claimed improvement must exceed transaction costs.

---

# 3. Previous research

## 3.1 The efficiency baseline

**Fama (1970)** established the efficient-market framework under which publicly available
price history should carry no exploitable predictive content. **Meese and Rogoff (1983)**
supplied the decisive empirical test for exchange rates specifically, finding that a random
walk outperformed structural models out of sample at horizons up to twelve months. Our
directional results — ROC-AUC ≈ 0.50 across every architecture tested — are **consistent
with** this literature rather than contradicting it, and we report them as such.

## 3.2 Volatility modelling

**Bollerslev (1986)** introduced the GARCH(1,1) specification that remains the standard
volatility benchmark. **Parkinson (1980)** derived the high-low range estimator used
throughout this project, which is approximately five times more efficient than squared
close-to-close returns. Our work uses GARCH(1,1) not as a competitor but as the *floor* that
any proposed model must clear.

## 3.3 Stochastic clocks

**Clark (1973)** proposed the subordinated-process model motivating this project's most
ambitious experiment. We tested it directly by training a liquid time-constant network whose
relaxation rate is input-dependent, then asking whether the learned clock tracks realised
transaction intensity. **It does not** (Section 7.3) — a negative result on Clark's
hypothesis as operationalised here, not on the theory itself.

## 3.4 Multiple testing and backtest overfitting

**Harvey, Liu and Zhu (2016)** argue that most published cross-sectional return predictors
would fail an honest multiple-testing correction. **López de Prado (2018)** develops the
same argument for backtest overfitting. This literature is the direct motivation for the
registry design in Section 6: without correction, unlimited testing against a near-zero
effect produces "significant" findings with near-certainty.

## 3.5 Microstructure noise

**Zhang, Mykland and Aït-Sahalia (2005)** show that realised volatility at high frequency is
dominated by microstructure noise rather than integrated variance. This underpins the
discrete-Hodge experiment (Section 5.4), where the observed quantity is provably pure
measurement error whose *amplitude* is nonetheless informative.

## 3.6 How this work differs

| Prior work | This project |
|---|---|
| Reports the model that worked | Reports all 56 hypotheses, including 37 rejections |
| Single significance threshold | Bonferroni-corrected per family, tightening as claims accumulate |
| Statistical significance as the criterion | Significance **and** a 0.92 bp transaction-cost floor |
| Complex model vs simple benchmark | Benchmark promoted to model when it wins (Section 7.1) |

**Four explicit comparisons** are made: against our own previous submission, against our own
previous headline claim, against an external pre-trained model (Kronos), and against the
incumbent production model — the last of which we lose.

---

# 4. Data acquisition, cleaning and validity

## 4.1 Sources and fallback chains

Two independent ingestion chains, neither of which fails hard:

- **Price:** MetaTrader 5 → yfinance → bundled historical CSV (`src/live_data.py`)
- **Macro:** FRED API → FRED public CSV → on-disk cache → `None` (`src/macro_data.py`)

The price-only model variant consumes no macro columns and is therefore immune to FRED
outages by construction, not by exception handling.

## 4.2 The daily research set

The modelled row set is **8,559 euro-era daily bars** (1999-01-05 → 2026-06-17), obtained
after feature engineering and NaN removal. Pre-1999 rows in the raw file are synthetic
padding and are excluded by every family. Splits are chronological throughout: train
`[0:70%]` = 5,991 rows, validation `[70%:80%]` = 856 rows, test `[80%:100%]` = 1,712 rows.

In [ ]:
# [INSERT CODE: load the daily research set and print split boundaries]
# from src.calendar_volatility import build_daily_dataset, chronological_masks
# Expected output: 8,559 rows | train 5,991 | validation 856 | test 1,712

## 4.3 High-frequency acquisition and two data bugs found before analysis

For the microstructure experiment, **eight years of M1 bars across six currency pairs**
(EUR/USD/JPY/GBP subgraph) were pulled from a retail broker: ~2.98 million bars per symbol,
**2,959,641 after a strict inner join** on common timestamps. Alignment loss was 0.59–0.77%
per pair.

Two defects were found and corrected *before* any analysis, both of which would have
invalidated results silently:

1. **Server clock misidentified.** The feed is CET/CEST, not the common EET default. The
   discriminator was a March excursion: the trading week opens at server hour 23 in every
   month except March, when the US has entered DST and the EU has not. A fixed-offset server
   shows no such excursion, and a whole-year aggregate hides it entirely. This could not
   fabricate the microstructure signal — a uniform clock shift cancels within the triangle
   identity — but it would have rotated every hour-of-day bucket by one, and hour-of-day is
   precisely what the falsification gate interrogates.

2. **Spread served in points, not price units.** MetaTrader 5 returns integer points. Used
   directly, the predicted bid-side offset inflates by a factor of 1/point — 100,000× on a
   five-digit major, turning a genuine ~0.22 bp offset into ~21,551 bp. Every triangle would
   have been flagged as a quote-convention error and the study would have halted on a units
   label.

Both are documented as data-provenance findings rather than quietly patched.

In [ ]:
# [INSERT CODE: M1 coverage report — src/curl_mt5_fetch.py output]
# Expected: per-symbol rows, first/last UTC timestamp, gap census, median tick_volume, median spread

## 4.4 Establishing that the activity proxy is real

MetaTrader 5's `tick_volume` counts price updates published by one broker's server. It is
not consolidated traded volume — FX has no central tape. Rather than assume it is
meaningful, we tested it.

Theory predicts $E[c^2] = \tfrac{1}{2} \cdot V / N$, where $V$ is summed edge variance and
$N$ a variance-weighted harmonic mean tick count, so fitting

$$\log E[c^2] = k + \alpha \log V + \beta \log N$$

should yield $\beta = -1$. Testing $\beta$ against $-1$ directly proved **invalid**: the
volatility term is measured with error and correlates with activity, so on honest synthetic
ticks the fit returns $\beta \approx -1.76$, not $-1.0$.

The valid test is a **permutation control** — recompute $\beta$ on shuffled tick counts,
destroying bar-level information while preserving the marginal distribution:

| Simulated feed | $\beta$ | $z$ vs shuffled |
|---|---:|---:|
| Honest | −1.76 | **−42.7** |
| 50% shuffled | −0.71 | −17.9 |
| Saturated at a ceiling | −4.26 | −35.6 |
| Constant | — | undefined (collinear) |
| Fully shuffled | +0.04 | **−0.6** |

The control correctly identifies an uninformative feed. This is the pattern used throughout
the project: **judge a statistic against what a true null produces in your own pipeline, not
against a textbook value.**

In [ ]:
# [INSERT CODE: tick_volume audit + permutation control]
# from src.curl_mt5 import tick_volume_audit, staleness_exponent_test
# Report: pct_at_max, n_distinct, yearly median ratio, beta, z_vs_shuffled

## 4.5 No-look-ahead guarantees

Four invariants are enforced by unit tests rather than convention:

1. Targets are constructed with `shift(-1)`; a window predicting row $t$ never contains data
   from $t+1$.
2. Forward-fill carries *past* values forward only, never future values backward.
3. Scalers and PCA are fitted on the training block exclusively and applied elsewhere by
   `.transform()`.
4. `TimeSeriesSplit` is used throughout; random K-fold appears nowhere.

In [ ]:
# [INSERT CODE: run the no-look-ahead test suite]
# python -m pytest -q tests/test_unit.py -k "lookahead or leakage or fred"

---

# 5. Methodology and architecture

All modelling code lives in `src/` as single-responsibility modules with docstrings stating
scope and constraints. Training and serving import the *same* `src/features.py`, making the
feature matrix byte-identical on both sides — research-to-production drift is structurally
impossible rather than merely discouraged.

## 5.1 Baseline: multi-task LSTM

A shared LSTM trunk with two heads — a linear return estimate and a sigmoid direction
probability — trained jointly. Five seeds are ensembled because single-seed runs showed
framework nondeterminism of the same magnitude as the effects under test.

## 5.2 The calendar model

$$\hat{\sigma}_{t+1} = f_{\text{dow}(t)} \cdot c \cdot \sqrt{\omega + \alpha r_t^2 + \beta h_t}$$

Three parameters from GARCH(1,1), one global scale, six weekday multipliers. Each stage is
fitted to minimise **MAE** — the family's registered metric — via a weighted median of
ratios, since a least-squares fit would optimise the wrong loss and be dragged by the fat
right tail of realised volatility. NumPy and pandas only; ten seconds to fit on a CPU.

## 5.3 Liquid time-constant network with spiking readout

Three components: a `BusinessTimeWarp` converting calendar $\Delta t$ to business $\Delta t$;
a closed-form continuous-time cell whose relaxation rate is input-dependent; and a leaky
integrate-and-fire readout in which *silence means abstention*, trained end-to-end with
surrogate gradients under a Lagrangian coverage floor.

## 5.4 Discrete-Hodge microstructure estimator

Log exchange rates form a 1-cochain on the graph of currencies. No-arbitrage makes it
**exact** — $\omega = d\varphi$ for a per-currency potential — so every triangle's curl
vanishes identically:

$$\log\tfrac{\text{EUR}}{\text{USD}} + \log\tfrac{\text{USD}}{\text{JPY}} + \log\tfrac{\text{JPY}}{\text{EUR}} = 0$$

Observed curl is therefore **pure measurement error**. Its amplitude scales as volatility ×
time-since-last-tick, making it a cross-sectional estimator of microstructure stress.

In [ ]:
# [INSERT CODE: calendar model definition and fit]
# from src.calendar_volatility import CalendarVolatilityModel
# Print the fitted parameters — all ten of them

In [ ]:
# [INSERT CODE: LTC + spiking architecture summary]
# from src.ltc_spiking_arch import LTCSpikingModel  (703 lines, JAX/Equinox)

---

# 6. Testing protocol

## 6.1 Three-block chronological split

`[0:70%]` fits every parameter. `[70%:80%]` is the **arbiter** for all hypothesis testing.
`[80%:100%]` is reserved for one-shot final reports and is explicitly declared *spent* for
feature search — it had previously been reused as a repeated selection criterion, which is
data snooping, and that error is documented rather than concealed.

## 6.2 Corrected significance

Each family maintains its own registry and its own bar $\alpha = 0.05 / \text{family size}$.
Registering a new claim tightens the bar for every claim in that family, retroactively. The
direction family currently sits at $\alpha = 0.05/9 \approx 0.00556$.

## 6.3 Nulls constructed rather than assumed

The project's most instructive methodological failure: hypothesis `H_ltc.2` asked whether a
learned clock tracks tick intensity. As originally specified it **could not fail** —
`log_tick_rate` was a network input, so untrained models already scored partial $\rho$ of
−0.42 to −0.71 and cleared the declared bar with no training whatsoever.

The repair had two parts. The model was made **tick-blind**, and the null became the
distribution over thirty untrained initialisations. That null then centred on
**+0.007 ± 0.038** — as a valid null must — and the trained model scored $z = +0.38$ against
a bar of $z < -3$.

The raw correlation was **−0.759**, large and in exactly the predicted direction. The
partial correlation was **+0.02**. The entire raw figure was a volatility confound, and
reporting it would have manufactured a discovery.

## 6.4 Block bootstrap and replication

Serially dependent data invalidates i.i.d. resampling, so all confidence intervals use
moving-block bootstraps with pre-registered block length. Where a claim clears, replication
on independent instruments is mandatory before it is believed.

## 6.5 Power analysis before spending significance

The G10 macro panel was **stopped before fitting**: its validation slice yields 29
independent observations against a pre-registered floor of 150, and the smallest resolvable
effect was 20.8 percentage points. Both hypotheses are recorded as REGISTERED-UNSPENT.
Reporting that data cannot answer a question is more honest than producing a confident null.

## 6.6 Unit testing

**452 test functions** covering no-look-ahead invariants, artifact checksums that prevent a
retrain from silently altering production models, and numerical parity between vectorised
and reference implementations.

In [ ]:
# [INSERT CODE: full test suite]
# python -m pytest -q     -> expected: 452 passed

In [ ]:
# [INSERT CODE: display the hypothesis registry]
# Load all results/*_hypothesis_log.csv, tabulate family / n / KEEP / DROP
# Expected: 15 families, 56 hypotheses, 37 DROP

---

# 7. Results

## 7.1 A ten-parameter model beats the neural ensemble

Scored on an identical row set (validation $n = 856$, test $n = 1{,}712$):

| Model | Val MAE | Val R² | Test MAE | Test R² |
|---|---:|---:|---:|---:|
| **Calendar (10 parameters)** | **0.16209** | **0.1602** | **0.19279** | **0.1348** |
| 5-seed multi-task LSTM ensemble | 0.18594 | 0.1444 | 0.21897 | 0.1098 |
| GARCH(1,1) | 0.20379 | 0.0094 | 0.23257 | 0.0356 |
| Persistence | 0.26133 | −0.8396 | 0.29214 | — |

It wins on **both** blocks with higher R² on both. The calendar component alone contributes
$\Delta$MAE **+0.02039** on the test block, CI95 **[+0.01685, +0.02409]**, excluding zero.

The entire fitted model:

```
GARCH  α = 0.0284   β = 0.9685   scale = 0.5495
Mon 1.291   Tue 1.232   Wed 1.271   Thu 1.354   Fri 0.275   Sun 1.058
```

The Friday multiplier of **0.275** is the weekend gap expressed as a single number.

**Honest decomposition.** The gain is not all calendar. An MAE-optimal scale applied to plain
GARCH already reaches 0.18405 on validation — level with the neural ensemble. The weekday
factors supply the remainder. Two inexpensive improvements, neither of them a neural network.

> ⚠ **Status.** The registry row reads `PENDING-PAIRED-CI`. The paired bootstrap against the
> ensemble's per-row predictions is implemented (`src/calendar_paired_bootstrap.py`) but must
> be executed before this is claimed as CLEARED. The GARCH here is a NumPy variance-targeting
> fit rather than the `arch` MLE used elsewhere, and the model was built and scored in a
> single pass with no pre-registration preceding measurement. All three caveats are recorded
> in the registry.

In [ ]:
# [INSERT CODE: reproduce the results table]
# from src.calendar_volatility import ...  -> validation + test comparison

In [ ]:
# [INSERT CODE: paired bootstrap vs the frozen ensemble]
# python -m src.calendar_paired_bootstrap   (requires TensorFlow)

## 7.2 Required visualisation 1 — the model itself

**Plot: weekday multipliers as a bar chart**, with a horizontal reference line at 1.0.
Friday at 0.275 should be visually unmistakable. Caption must state that the target is
$|r_{t+1}|$, so Friday's low value reflects the *weekend* target, not low Friday activity —
without that caption the chart misleads.

**Plot: predicted vs realised volatility over the test block**, calendar model and ensemble
overlaid on the same axes.

**Plot: fitted GARCH conditional volatility with the weekday multiplier applied**, one year
of the test block, so the sawtooth calendar structure is visible.

In [ ]:
# [INSERT CODE: weekday factor bar chart + predicted-vs-realised overlay]

## 7.3 The liquid time-constant experiment — three honest negatives

| Hypothesis | Verdict |
|---|---|
| `H_ltc.1` selective CfC+LIF vs GARCH×weekday | CLEARED — marginal, fragile, **mechanism unsupported** |
| `H_ltc.2` learned clock tracks tick rate | **DROP** ($z = +0.38$ against a bar of $z < -3$) |
| `H_spk.1` trained readout beats a fixed σ threshold | **DROP** (failed replication) |

`H_ltc.1` cleared arithmetically and was **not promoted**. The model that passed had a
collapsed time warp — weekend state retention $2.8 \times 10^{-9}$, meaning it erased its
hidden state every weekend. The mechanism the family exists to test was absent from the model
that passed the test. Its clearance was also fragile to block length, holding at 24 bars but
not at 96 or 168 — and the latter two correspond to the four-day and weekly structure that
FX actually exhibits.

`H_spk.1` was tested on two independent instruments with a pre-registered requirement that
both must clear. GBPUSD gave $\Delta$AURC **−0.000752** (wrong direction); AUDUSD gave
**+0.017948**. One instrument is an anecdote. **DROP.**

The diagnostic explaining the split is more useful than the verdict: where the model's
uncertainty head is well calibrated (GBPUSD, $\rho = +0.37$ with realised error), the
spiking readout converges to 90% correlation with it and adds nothing. Where that head is
broken (AUDUSD, $\rho = -0.08$; EURUSD, $\rho = -0.18$), the readout diverges and appears
impressive. **The spiking readout is not a superior confidence estimator; it is a rescue
mechanism for a miscalibrated one.**

## 7.4 Required visualisation 2 — risk-coverage

**Plot: risk-coverage curves**, four rankings on one axis — LIF membrane, negative predicted
σ, negative |μ|, and random. This single figure carries the section's argument:

| Ranking | AURC (lower is better) |
|---|---:|
| LIF membrane | 0.029627 |
| Random | 0.037637 |
| −σ (the model's own uncertainty head) | 0.045060 |
| −\|μ\| | 0.045298 |

The model's own uncertainty estimate ranks **worse than random** — it is most confident
precisely where it is most wrong.

**Plot: hour-of-day histogram of the readout's most-confident decile against the base rate.**
77.5% of that decile falls in 21:00–05:00 UTC against a 37.5% base rate, with 5.68× lift at
midnight and *zero* firing during London and New York hours. The readout learned "predict
when the market is quiet" — an hour-of-day rule.

Decisively: ranking by a day×hour lookup table gives AURC **0.025753**, beating the entire
spiking architecture at 0.029627.

In [ ]:
# [INSERT CODE: risk-coverage curves and hour-of-day concentration histogram]

## 7.5 Auditing the live system

Every result above concerns research code. This one concerns the deployed path.

| | Validation `[70:80]` | Test `[80:100]` |
|---|---:|---:|
| Accuracy lift from the 0.52 confidence guard | **+10.5 pp** | −0.005 |
| ROC-AUC on the ≥0.52 subset | 0.638 | 0.501 |
| Reliability table monotonic | yes | no |

**The validation panel measures memorisation.** That slice sits inside the gradient-boosting
model's own training range, so the guard cannot be evaluated there. On the honest block every
apparent benefit vanishes: twelve out-of-sample rank correlations between −0.025 and +0.047,
none significant.

The guard is **inert, not harmful**. Its genuine function is display honesty — preventing the
dashboard from presenting a coin flip as an agreement — and documentation should not imply it
filters for accuracy.

This panel is retained deliberately, because it shows what an in-sample calibration audit of
this system looks like: convincing, monotone, and entirely spurious.

## 7.6 Required visualisation 3 — reliability

**Plot: reliability diagram, validation and test side by side**, same axes, with the diagonal
marked and bin counts annotated. The validation panel is smoothly monotone; the test panel is
not. Presenting them together is the honest presentation — showing only validation would be
the misleading one.

**Plot: Brier decomposition** (calibration / resolution / uncertainty) as a stacked bar per
variant and slice, with the trivial constant predictor marked. Uncertainty (0.2499) dwarfs
both other components by roughly 100×, which is the visual statement that these probabilities
are well-behaved because they barely move.

In [ ]:
# [INSERT CODE: reliability diagrams and Brier decomposition]
# python -m src.calibration_audit

## 7.7 Instrument validation before belief

The discrete-Hodge estimator was validated on **synthetic data where ground truth is known**
before being applied to real markets:

| Check | Result |
|---|---|
| Exact simultaneity ⇒ zero curl | max \|curl\| < 1e-12 |
| Closed-form null vs realised variance | ratio 0.87–0.98 |
| Curl variance across M1→H1 (theory: flat) | 0.94 / 0.86 / 0.91 / 0.81 bp |
| Return variance across M1→H1 (theory: ∝ Δ) | 3.4 / 8.1 / 14.1 / 19.2 bp |
| Injected 2 bp dislocation, detection AUC | 0.996 |
| Deliberately injected convention bug | constant 30.5 bp — caught |

A **stopping rule** was written before the real-data run, with the expected outcome stated in
advance as "nothing there". Committing to a falsification criterion before seeing data is the
only defence against finding what one went looking for.

---

# 8. Conclusion

## 8.1 What was found

Across fifty-six registered hypotheses, EUR/USD **direction** remains unpredictable at
ROC-AUC ≈ 0.50. This is reported as a finding consistent with Meese and Rogoff (1983), not
concealed as a failure.

**Volatility** is predictable — and the best predictor found is a ten-parameter calendar
model that outperforms a trained five-seed neural ensemble on both evaluation blocks. Its six
weekday multipliers encode the weekend-gap effect more accurately than the network that
discovered it.

## 8.2 Why the simple model won

The neural ensemble had to *learn* the calendar from data, spending capacity on a
deterministic, known structure. The calendar model is *told* the calendar and spends its
three remaining parameters on volatility clustering. When the exploitable structure is small
and known in advance, encoding it directly dominates learning it.

The same explanation covers the spiking readout. It appeared to identify its own confident
moments; a day×hour lookup table encoded the same information more accurately (AURC 0.025753
versus 0.029627). The network was rediscovering the intraday liquidity cycle, imperfectly.

## 8.3 What the discipline bought

Three findings would have been published as discoveries by a less careful process:

1. A volatility ensemble beating GARCH — mechanism revealed as a weekday effect.
2. A learned clock correlating −0.759 with tick intensity — entirely a volatility confound;
   the partial correlation is +0.02.
3. A selective readout with a 39% risk reduction on its confident decile — 0.29 bp of value
   against a 0.92 bp cost, and concentrated in the Asian session.

Each was caught by a control constructed *before* the result was believed.

## 8.4 Limitations

- The calendar model's paired confidence interval against the ensemble is implemented but not
  yet executed; the registry row reads `PENDING-PAIRED-CI`.
- Its GARCH is a NumPy variance-targeting fit rather than the `arch` MLE used elsewhere.
- It was built and scored in a single pass, without pre-registration preceding measurement.
- Four macro features are retained as KEEP-provisional, none having cleared the corrected bar.
- The discrete-Hodge work is validated on synthetic data only; no hypothesis is registered.

## 8.5 Closing

The project's deliverable is not a forecast. It is an apparatus that can distinguish a
finding from an artifact, applied honestly enough to reject thirty-seven of its own fifty-six
hypotheses — including several the author wanted to be true.

In a near-efficient market, knowing precisely what does not work, and being able to
demonstrate that it was tested fairly, is the result.

---

# Appendix A — Rubric map

| Criterion | Where addressed | Evidence |
|---|---|---|
| **Problem statement (10)** | §2 | Both operational questions defined; formal target specification; 0.92 bp economic threshold applied as a hard filter |
| **Layout (20)** | throughout | Eight numbered sections, reader's guide, LaTeX for all formulations, consistent theory → code → interpretation structure |
| **Code quality (20)** | §5, `src/` | 52 single-responsibility modules, typed and linted, single-source-of-truth feature contract, 452 tests |
| **Previous research (10)** | §3 | Eleven cited sources; four distinct comparisons — own prior submission, own prior claim, external model, incumbent production model |
| **Data (10)** | §4 | Dual fallback chains, 2.96M aligned M1 bars, two provenance bugs found pre-analysis, permutation-validated activity proxy, four test-enforced no-look-ahead invariants |
| **Testing (10)** | §6 | 452 unit tests, 56 registered hypotheses, Bonferroni correction, permutation and initialisation nulls, block bootstrap, cross-instrument replication, pre-fit power analysis |
| **Visualisation (10)** | §7.2, §7.4, §7.6 | Risk-coverage curves, reliability diagrams, Brier decomposition, weekday factors, hour-of-day concentration — each captioned with what it shows and what it does not |
| **Communication (10)** | §1, §8 | Narrative arc from assumption through falsification to an honest negative; limitations stated before conclusions |

# Appendix B — Reproducibility

```bash
pip install -r requirements.txt          # no TensorFlow or JAX needed for the calendar model
python -m pytest -q                      # 452 tests
python -m src.calendar_volatility        # the headline model
python -m src.calibration_audit          # §7.5
python -m src.calendar_paired_bootstrap  # requires TensorFlow
```

Pre-trained artifacts and the feature matrix are committed; no training is required to
reproduce any result in this notebook.

# Appendix C — Registry

Fifteen `results/*_hypothesis_log.csv` files, append-only, each row carrying its date,
arbiter, point estimate, confidence interval, corrected α, verdict and full notes — including
disclosures where protocol was imperfect.